week 5


In [1]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def get_neighbors(state, maze):
    x, y = state
    moves = [(1,0), (-1,0), (0,1), (0,-1)]
    neighbors = []
    for dx, dy in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < len(maze) and 0 <= ny < len(maze[0]) and maze[nx][ny] == 0:
            neighbors.append((nx, ny))
    return neighbors


In [2]:

def rbfs(state, goal, maze, g, bound, path, expanded):
    f = g + manhattan(state, goal)

    if state == goal:
        return path, f

    expanded[0] += 1
    successors = []

    for s in get_neighbors(state, maze):
        if s not in path:
            successors.append([s, g + 1, g + 1 + manhattan(s, goal)])

    if not successors:
        return None, float('inf')

    while True:
        successors.sort(key=lambda x: x[2])
        best = successors[0]

        if best[2] > bound:
            return None, best[2]

        alternative = successors[1][2] if len(successors) > 1 else float('inf')

        result, best[2] = rbfs(
            best[0],
            goal,
            maze,
            best[1],
            min(bound, alternative),
            path + [best[0]],
            expanded
        )

        if result is not None:
            return result, best[2]


In [3]:
maze = [
    [0,0,0,1,0],
    [1,1,0,1,0],
    [0,0,0,0,0],
    [0,1,1,1,0],
    [0,0,0,0,0]
]

start = (0,0)
goal = (4,4)

expanded = [0]
path, cost = rbfs(start, goal, maze, 0, float('inf'), [start], expanded)

print("RBFS Path:", path)
print("Nodes Expanded:", expanded[0])


RBFS Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
Nodes Expanded: 8


In [4]:
def ida_star(start, goal, maze):
    threshold = manhattan(start, goal)
    expanded = [0]

    def dfs(state, g, threshold, path):
        f = g + manhattan(state, goal)
        if f > threshold:
            return f
        if state == goal:
            return path

        expanded[0] += 1
        min_cost = float('inf')

        for s in get_neighbors(state, maze):
            if s not in path:
                result = dfs(s, g + 1, threshold, path + [s])
                if isinstance(result, list):
                    return result
                min_cost = min(min_cost, result)

        return min_cost

    while True:
        result = dfs(start, 0, threshold, [start])
        if isinstance(result, list):
            return result, expanded[0]
        threshold = result


In [5]:
path, expanded = ida_star(start, goal, maze)
print("IDA Path:", path)
print("Nodes Expanded:", expanded)


IDA* Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
Nodes Expanded: 8


In [6]:
import heapq

def sma_star(start, goal, maze, memory_limit):
    open_list = []
    heapq.heappush(open_list, (manhattan(start, goal), 0, start, [start]))
    expanded = 0

    while open_list:
        if len(open_list) > memory_limit:
            open_list.sort(reverse=True)
            open_list.pop(0)

        f, g, state, path = heapq.heappop(open_list)

        if state == goal:
            return path, expanded

        expanded += 1

        for s in get_neighbors(state, maze):
            if s not in path:
                new_g = g + 1
                new_f = new_g + manhattan(s, goal)
                heapq.heappush(open_list, (new_f, new_g, s, path + [s]))

    return None, expanded


In [7]:
path, expanded = sma_star(start, goal, maze, memory_limit=10)
print("SMA Path:", path)
print("Nodes Expanded:", expanded)


SMA* Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
Nodes Expanded: 8
